# Median Intertidal Surface by AOI

## Overview

This script computes the **pixel-by-pixel median surface** for each Area of Interest (AOI) using all available yearly raster surfaces. The goal is to obtain a representative **median intertidal morphology** from the time series of generated surfaces.

The script is designed to run in **Google Colab** and access data stored in a **Google Drive Shared Drive**.

---

## Input Data

The script expects the following folder structure:

├── 01_AOI01/

│ └── 04_SUPERFICIES/

│ ├── 1985/

│ │ └── surface.tif

│ ├── 1986/

│ └── ...

├── 02_AOI02/

└── 03_AOI03/

Each yearly folder should contain a **GeoTIFF surface** generated from the waterline method.

---

## Processing Steps

For each AOI, the script:

1. Accesses the folder `04_SUPERFICIES`.
2. Finds all raster files (`.tif` / `.tiff`) across yearly subfolders.
3. Loads the rasters into a stack.
4. Converts `nodata` values to `NaN`.
5. Computes the **pixel-by-pixel median** using all available rasters.
6. Exports a new GeoTIFF representing the **median surface**.

---

## Output

For each AOI, the script generates one raster:

- 01_AOI_1_median_surface.tif
- 02_AOI_2_median_surface.tif
- 03_AOI_3_median_surface.tif

## Notes

All rasters within an AOI must have:

- the same spatial resolution  
- the same CRS  
- the same grid alignment  

Otherwise, preprocessing (reprojection or resampling) will be required before computing the median.

---

## Application

The resulting median surface provides a **representation of typical intertidal morphology**, reducing the influence of outliers and temporal noise in individual yearly surfaces.

Instalar bibliotecas

In [35]:
#!pip install -q rasterio numpy

import os
import glob
import math
import numpy as np
import rasterio
from collections import Counter
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_origin
from google.colab import drive
from collections import Counter
from rasterio.crs import CRS

Montar Google Drive

In [22]:
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Definindo pastas

In [29]:
ROOT = "/content/drive/Shareddrives/Batimetrias_Babitonga"

AOIS = [
    "01_AOI01",
    "02_AOI02",
    "03_AOI03"
]

OUTPUT_DIR = os.path.join(ROOT, "_MEDIAN_SURFACES")
os.makedirs(OUTPUT_DIR, exist_ok=True)


Função para encontrar rasters de um AOI

In [30]:
def find_surface_rasters(aoi_dir):
    surf_dir = os.path.join(aoi_dir, "04_SUPERFICIES")
    if not os.path.isdir(surf_dir):
        return []

    tif_files = glob.glob(os.path.join(surf_dir, "**", "*.tif"), recursive=True)
    tiff_files = glob.glob(os.path.join(surf_dir, "**", "*.tiff"), recursive=True)

    return sorted(tif_files + tiff_files)

Detectar CRS dominante

In [42]:
def get_dominant_crs(raster_paths):
    crs_list = []

    for path in raster_paths:
        try:
            with rasterio.open(path) as src:
                if src.crs is not None:
                    crs_list.append(src.crs.to_string())
        except Exception as e:
            print(f"[AVISO] Erro ao ler CRS de {path}: {e}")

    if not crs_list:
        print("  [AVISO] Nenhum raster com CRS definido. Assumindo EPSG:4326")
        return CRS.from_epsg(4326)

    dominant = Counter(crs_list).most_common(1)[0][0]
    return CRS.from_string(dominant)

Definir grade comum (união total)

In [43]:
def build_common_grid_union(raster_paths, target_crs):
    """
    Cria grade comum usando:
    - resolução do primeiro raster disponível
    - união total das extensões
    - fallback de CRS quando necessário
    """

    if not raster_paths:
        raise ValueError("Nenhum raster fornecido.")

    xres = None
    yres = None
    all_bounds = []

    for i, path in enumerate(raster_paths):
        with rasterio.open(path) as src:
            # pega resolução do primeiro raster, mesmo sem CRS
            if xres is None:
                xres = abs(src.transform.a)
                yres = abs(src.transform.e)

            src_crs = src.crs if src.crs is not None else target_crs

            # se o raster já está no CRS alvo (ou assumido nele), usa bounds direto
            if src_crs == target_crs:
                b = src.bounds
                all_bounds.append((b.left, b.bottom, b.right, b.top))
            else:
                b = transform_bounds(src_crs, target_crs, *src.bounds, densify_pts=21)
                all_bounds.append(b)

    if xres is None or yres is None:
        raise ValueError("Não foi possível determinar a resolução dos rasters.")

    minx = min(b[0] for b in all_bounds)
    miny = min(b[1] for b in all_bounds)
    maxx = max(b[2] for b in all_bounds)
    maxy = max(b[3] for b in all_bounds)

    width = int(math.ceil((maxx - minx) / xres))
    height = int(math.ceil((maxy - miny) / yres))

    transform = from_origin(minx, maxy, xres, yres)

    return {
        "crs": target_crs,
        "transform": transform,
        "width": width,
        "height": height,
        "xres": xres,
        "yres": yres
    }

Ler raster e reprojetar para grade comum

In [44]:
def read_to_common_grid(path, common_grid, fallback_crs=None):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        src_nodata = src.nodata
        src_crs = src.crs if src.crs is not None else fallback_crs

        if src_crs is None:
            raise ValueError(f"Raster sem CRS e sem fallback definido: {path}")

        if src.nodata is not None:
            arr[arr == src_nodata] = np.nan

        dst_arr = np.full(
            (common_grid["height"], common_grid["width"]),
            np.nan,
            dtype="float32"
        )

        reproject(
            source=arr,
            destination=dst_arr,
            src_transform=src.transform,
            src_crs=src_crs,
            dst_transform=common_grid["transform"],
            dst_crs=common_grid["crs"],
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear
        )

    return dst_arr


Função para calcular mediana pixel a pixel

In [45]:
def median_raster_union(raster_paths, out_median, out_count):
    target_crs = get_dominant_crs(raster_paths)
    print(f"  CRS de referência: {target_crs}")

    common_grid = build_common_grid_union(raster_paths, target_crs)

    arrays = []
    used_paths = []

    for path in raster_paths:
        try:
            with rasterio.open(path) as src:
                if src.crs is None:
                    print(f"  [AVISO] CRS ausente em {os.path.basename(path)} -> assumindo {target_crs}")

            arr = read_to_common_grid(path, common_grid, fallback_crs=target_crs)
            arrays.append(arr)
            used_paths.append(path)

        except Exception as e:
            print(f"  [AVISO] Raster ignorado: {os.path.basename(path)} | motivo: {e}")

    if not arrays:
        raise ValueError("Nenhum raster pôde ser reprojetado/processado.")

    stack = np.stack(arrays, axis=0)

    median_arr = np.nanmedian(stack, axis=0).astype("float32")
    count_arr = np.sum(~np.isnan(stack), axis=0).astype("int16")

    out_nodata = -9999.0
    median_out = median_arr.copy()
    median_out[np.isnan(median_out)] = out_nodata

    profile = {
        "driver": "GTiff",
        "height": common_grid["height"],
        "width": common_grid["width"],
        "count": 1,
        "dtype": "float32",
        "crs": common_grid["crs"],
        "transform": common_grid["transform"],
        "compress": "lzw",
        "nodata": out_nodata
    }

    with rasterio.open(out_median, "w", **profile) as dst:
        dst.write(median_out, 1)

    count_profile = profile.copy()
    count_profile.update(dtype="int16", nodata=0)

    with rasterio.open(out_count, "w", **count_profile) as dst:
        dst.write(count_arr, 1)

    print(f"  {len(used_paths)} rasters usados no cálculo.")

# ============================================
# 10. Processar AOIs
# ============================================
for aoi in AOIS:
    aoi_dir = os.path.join(ROOT, aoi)

    if not os.path.isdir(aoi_dir):
        print(f"[AVISO] AOI não encontrada: {aoi_dir}")
        continue

    raster_list = find_surface_rasters(aoi_dir)

    if not raster_list:
        print(f"[AVISO] Nenhum raster encontrado para {aoi}")
        continue

    print(f"\n{aoi}: {len(raster_list)} rasters encontrados")

    out_median = os.path.join(OUTPUT_DIR, f"{aoi}_median_surface.tif")
    out_count  = os.path.join(OUTPUT_DIR, f"{aoi}_valid_count.tif")

    try:
        median_raster_union(raster_list, out_median, out_count)
        print(f"[OK] Mediana salva em: {out_median}")
        print(f"[OK] Contagem salva em: {out_count}")
    except Exception as e:
        print(f"[ERRO] {aoi}: {e}")

print("\nProcessamento concluído.")


01_AOI01: 34 rasters encontrados
  CRS de referência: EPSG:4326
  [AVISO] CRS ausente em dup_delta_1995.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_1996.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_1997.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_1999.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2000.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2001.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2004.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2005.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2006.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2007.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2009.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2010.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2011.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2012.tif -> assumindo EPSG:43

/tmp/ipykernel_270/1121596825.py:28: RuntimeWarning: All-NaN slice encountered
  median_arr = np.nanmedian(stack, axis=0).astype("float32")


  34 rasters usados no cálculo.
[OK] Mediana salva em: /content/drive/Shareddrives/Batimetrias_Babitonga/_MEDIAN_SURFACES/01_AOI01_median_surface.tif
[OK] Contagem salva em: /content/drive/Shareddrives/Batimetrias_Babitonga/_MEDIAN_SURFACES/01_AOI01_valid_count.tif

02_AOI02: 33 rasters encontrados
  [AVISO] Nenhum raster com CRS definido. Assumindo EPSG:4326
  CRS de referência: EPSG:4326
  [AVISO] CRS ausente em sup_aterro_1985.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_aterro_1986.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_aterro_1991.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_aterro_1992.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_aterro_1994.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_aterro_1995.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_aterro_1996.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_aterro_1997.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_aterro_1998.tif -> assumindo EPSG:

/tmp/ipykernel_270/1121596825.py:28: RuntimeWarning: All-NaN slice encountered
  median_arr = np.nanmedian(stack, axis=0).astype("float32")


  33 rasters usados no cálculo.
[OK] Mediana salva em: /content/drive/Shareddrives/Batimetrias_Babitonga/_MEDIAN_SURFACES/02_AOI02_median_surface.tif
[OK] Contagem salva em: /content/drive/Shareddrives/Batimetrias_Babitonga/_MEDIAN_SURFACES/02_AOI02_valid_count.tif

03_AOI03: 36 rasters encontrados
  CRS de referência: EPSG:4326
  [AVISO] CRS ausente em sup_bsul_1985.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_bsul_1986.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_bsul_1987.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_bsul_1988.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_bsul_1990.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_bsul_1997.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_bsul_1999.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_bsul_2000.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_bsul_2001.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_bsul_2002.tif -> assumindo EPSG:4326
  [AVISO]

/tmp/ipykernel_270/1121596825.py:28: RuntimeWarning: All-NaN slice encountered
  median_arr = np.nanmedian(stack, axis=0).astype("float32")
